## 2 sep code

In [ ]:
import bpy
import os
import math

# ============================================================
# SAFETY: ENSURE OBJECT MODE
# ============================================================
if bpy.context.object and bpy.context.object.mode != 'OBJECT':
    bpy.ops.object.mode_set(mode='OBJECT')

# ============================================================
# FIND OBJECTS FROM YOUR SCENE
# ============================================================
soil = bpy.data.objects.get("SOIL_GROUND")
nilgai = bpy.data.objects.get("mesh_0")
grass_objects = []

for obj in bpy.context.scene.objects:
    if obj.type == 'MESH' and obj.name.startswith("Object_2"):
        grass_objects.append(obj)

if soil is None:
    raise RuntimeError("SOIL_GROUND not found.")
if nilgai is None:
    raise RuntimeError("mesh_0 not found.")

# ============================================================
# ADJUST THE GROUND SIZE
# ============================================================
soil.scale = (2.0, 2.0, 1.0) 

# ============================================================
# FIX GRASS ORIENTATION & ARRANGE IN GRID (WITH DUPLICATION)
# ============================================================
if len(grass_objects) > 0:
    bpy.ops.object.select_all(action='DESELECT')
    
    # 1. Keep the first grass as a template and delete the rest
    base_grass = grass_objects[0]
    for i in range(1, len(grass_objects)):
        try:
            bpy.data.objects.remove(grass_objects[i], do_unlink=True)
        except:
            pass
            
    base_grass.select_set(True)
    bpy.context.view_layer.objects.active = base_grass
    bpy.ops.object.parent_clear(type='CLEAR_KEEP_TRANSFORM')
    bpy.ops.object.transform_apply(location=False, rotation=True, scale=True)
    bpy.ops.object.select_all(action='DESELECT')
    
    # ==========================================
    # SET YOUR GRASS COUNT AND SPACING HERE
    # ==========================================
    grass_rows = 10       
    grass_columns = 10    
    spacing = 1.25 
    # ==========================================
    
    start_x = -((grass_columns - 1) * spacing) / 2
    start_y = -((grass_rows - 1) * spacing) / 2
    
    grass_objects = [base_grass] 
    
    for row in range(grass_rows):
        for col in range(grass_columns):
            if row == 0 and col == 0:
                current_grass = base_grass
            else:
                current_grass = base_grass.copy()
                bpy.context.collection.objects.link(current_grass)
                grass_objects.append(current_grass)
            
            new_x = start_x + (col * spacing)
            new_y = start_y + (row * spacing)
            
            current_grass.matrix_world.translation.x = new_x
            current_grass.matrix_world.translation.y = new_y
            current_grass.matrix_world.translation.z = 0.0
            
# ============================================================
# CREATE MATERIALS (THERMAL AND NATURAL)
# ============================================================
def create_material(name, color, roughness=0.8, is_thermal=True):
    mat = bpy.data.materials.get(name)
    if mat is None:
        mat = bpy.data.materials.new(name)
    mat.use_nodes = True
    
    nodes = mat.node_tree.nodes
    bsdf = nodes.get("Principled BSDF")
    
    if bsdf:
        bsdf.inputs["Base Color"].default_value = (color[0], color[1], color[2], 1)
        bsdf.inputs["Roughness"].default_value = roughness
        
        # Only add emission glow if it is a thermal material
        if is_thermal:
            if "Emission Color" in bsdf.inputs:
                bsdf.inputs["Emission Color"].default_value = (color[0], color[1], color[2], 1)
            if "Emission Strength" in bsdf.inputs:
                bsdf.inputs["Emission Strength"].default_value = 0.15
        else:
            if "Emission Strength" in bsdf.inputs:
                bsdf.inputs["Emission Strength"].default_value = 0.0
            
    return mat

# We still create these so the 2nd script can find them later
gray_animal = create_material("THERMAL_GRAY_ANIMAL", (0.95, 0.95, 0.95))
gray_grass = create_material("THERMAL_GRAY_GRASS", (0.22, 0.22, 0.22))
gray_soil = create_material("THERMAL_GRAY_SOIL", (0.10, 0.10, 0.10))

ironbow_animal = create_material("THERMAL_IRONBOW_ANIMAL", (1.0, 0.10, 0.01))
ironbow_grass = create_material("THERMAL_IRONBOW_GRASS", (0.01, 0.02, 0.20))
ironbow_soil = create_material("THERMAL_IRONBOW_SOIL", (0.02, 0.01, 0.08))

# Create Natural Colors for the GUI Preview
natural_grass = create_material("NATURAL_GRASS", (0.05, 0.25, 0.02), is_thermal=False)
natural_soil = create_material("NATURAL_SOIL", (0.15, 0.10, 0.05), is_thermal=False)

def assign_material(obj, material):
    if obj is None: return
    if len(obj.data.materials) == 0:
        obj.data.materials.append(material)
    else:
        obj.data.materials[0] = material

# ============================================================
# FIXED CAMERA
# ============================================================
camera = bpy.data.objects.get("THERMAL_FIXED_CAMERA")
if camera is None:
    camera_data = bpy.data.cameras.new("THERMAL_FIXED_CAMERA")
    camera = bpy.data.objects.new("THERMAL_FIXED_CAMERA", camera_data)
    bpy.context.collection.objects.link(camera)

camera.location = (8.5, -12.0, 2.5)
camera.rotation_euler = (math.radians(85), 0, math.radians(35))
camera.data.lens = 52
bpy.context.scene.camera = camera

# ============================================================
# FIXED LIGHT (CHANGED TO SUNLIGHT FOR NATURAL LOOK)
# ============================================================
light = bpy.data.objects.get("THERMAL_FIXED_LIGHT")
if light is None:
    light_data = bpy.data.lights.new("THERMAL_FIXED_LIGHT", type='SUN')
    light = bpy.data.objects.new("THERMAL_FIXED_LIGHT", light_data)
    bpy.context.collection.objects.link(light)

# Configure as a Sun so it lights the whole environment evenly
light.data.type = 'SUN'
light.data.energy = 2.5
light.data.angle = math.radians(11)
light.rotation_euler = (math.radians(45), math.radians(30), math.radians(45))

# ============================================================
# REALISTIC BLUE SKY BACKGROUND
# ============================================================
world = bpy.data.worlds.get("ThermalWorld")
if world is None:
    world = bpy.data.worlds.new("ThermalWorld")

bpy.context.scene.world = world
world.use_nodes = True
nodes = world.node_tree.nodes
links = world.node_tree.links

for n in nodes:
    nodes.remove(n)

node_out = nodes.new('ShaderNodeOutputWorld')
node_bg = nodes.new('ShaderNodeBackground')

# Slightly softer blue so it doesn't overpower the green grass
node_bg.inputs['Color'].default_value = (0.15, 0.45, 0.85, 1.0) 
node_bg.inputs['Strength'].default_value = 0.8

links.new(node_bg.outputs['Background'], node_out.inputs['Surface'])

for area in bpy.context.screen.areas:
    if area.type == 'VIEW_3D':
        for space in area.spaces:
            if space.type == 'VIEW_3D':
                space.shading.use_scene_world = True

# ============================================================
# RENDER SETTINGS
# ============================================================
scene = bpy.context.scene
scene.render.engine = 'BLENDER_EEVEE'
scene.render.resolution_x = 640
scene.render.resolution_y = 480
scene.render.resolution_percentage = 100
scene.render.image_settings.file_format = 'PNG'
scene.render.film_transparent = False

# ============================================================
# PREPARE NATURAL MATERIALS (DO NOT TOUCH NILGAI)
# ============================================================

# We DO NOT assign anything to the nilgai here, so it keeps its original textures!
assign_material(soil, natural_soil)

for obj in grass_objects:
    assign_material(obj, natural_grass)

print("\n========================================")
print("MAIN SCENE READY (NATURAL COLORS APPLIED)")
print("========================================")

In [ ]:
import bpy
import os
import math
import csv


# ============================================================
# DATASET SETTINGS
# ============================================================

OUTPUT_FOLDER = os.path.join(
    bpy.path.abspath("//"),
    "dataset_grayscale_ironbow"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)


# ============================================================
# FIND OBJECTS
# ============================================================

soil = bpy.data.objects.get(
    "SOIL_GROUND"
)

nilgai = bpy.data.objects.get(
    "mesh_0"
)

grass_objects = []

for obj in bpy.context.scene.objects:

    if obj.type == 'MESH' and obj.name.startswith("Object_2"):

        grass_objects.append(obj)


print("\n========================================")
print("PIXEL VALUE DATASET GENERATOR")
print("========================================")

print("Soil:", soil.name if soil else "NOT FOUND")
print("Nilgai:", nilgai.name if nilgai else "NOT FOUND")
print("Grass:", len(grass_objects))


if soil is None:
    raise RuntimeError(
        "SOIL_GROUND not found."
    )

if nilgai is None:
    raise RuntimeError(
        "mesh_0 not found."
    )


# ============================================================
# GET MATERIALS
# ============================================================

gray_animal = bpy.data.materials.get(
    "THERMAL_GRAY_ANIMAL"
)

gray_grass = bpy.data.materials.get(
    "THERMAL_GRAY_GRASS"
)

gray_soil = bpy.data.materials.get(
    "THERMAL_GRAY_SOIL"
)

ironbow_animal = bpy.data.materials.get(
    "THERMAL_IRONBOW_ANIMAL"
)

ironbow_grass = bpy.data.materials.get(
    "THERMAL_IRONBOW_GRASS"
)

ironbow_soil = bpy.data.materials.get(
    "THERMAL_IRONBOW_SOIL"
)


if gray_animal is None:
    raise RuntimeError(
        "Run 01_GUI_Main_Scene.py first."
    )


# ============================================================
# ASSIGN MATERIAL
# ============================================================

def assign_material(obj, material):

    if obj is None:
        return

    if material is None:
        return

    if len(obj.data.materials) == 0:

        obj.data.materials.append(
            material
        )

    else:

        obj.data.materials[0] = material


# ============================================================
# MATERIAL SETUP
# ============================================================

def set_grayscale():

    print(
        "Applying GRAYSCALE / WHITE-HOT materials"
    )

    assign_material(
        nilgai,
        gray_animal
    )

    assign_material(
        soil,
        gray_soil
    )

    for obj in grass_objects:

        assign_material(
            obj,
            gray_grass
        )


def set_ironbow():

    print(
        "Applying IRONBOW materials"
    )

    assign_material(
        nilgai,
        ironbow_animal
    )

    assign_material(
        soil,
        ironbow_soil
    )

    for obj in grass_objects:

        assign_material(
            obj,
            ironbow_grass
        )


# ============================================================
# STORE ORIGINAL NILGAI TRANSFORM
# ============================================================

original_location = (
    nilgai.location.copy()
)

original_rotation = (
    nilgai.rotation_euler.copy()
)

original_scale = (
    nilgai.scale.copy()
)


# ============================================================
# ANIMAL POSITIONS
# ============================================================

# CAMERA DOES NOT MOVE.
#
# ONLY NILGAI ROTATES.
#
# Positive values = one direction
# Negative values = opposite direction

poses = {

    "left_1": math.radians(20),

    "left_2": math.radians(45),

    "right_1": math.radians(-20),

    "right_2": math.radians(-45)

}


# ============================================================
# PIXEL CSV FUNCTION
# ============================================================

def save_pixel_values(
    image_name,
    image
):

    csv_name = (
        os.path.splitext(image_name)[0]
        + "_pixels.csv"
    )

    csv_path = os.path.join(
        OUTPUT_FOLDER,
        csv_name
    )

    print(
        "Extracting pixels:",
        image_name
    )

    width = image.size[0]
    height = image.size[1]

    with open(
        csv_path,
        "w",
        newline=""
    ) as csv_file:

        writer = csv.writer(
            csv_file
        )

        writer.writerow([
            "x",
            "y",
            "R",
            "G",
            "B",
            "A",
            "Gray_Value"
        ])

        for y in range(height):

            for x in range(width):

                pixel_index = (
                    y * width + x
                ) * 4

                r = image.pixels[
                    pixel_index
                ]

                g = image.pixels[
                    pixel_index + 1
                ]

                b = image.pixels[
                    pixel_index + 2
                ]

                a = image.pixels[
                    pixel_index + 3
                ]

                gray_value = (
                    0.299 * r
                    + 0.587 * g
                    + 0.114 * b
                )

                writer.writerow([
                    x,
                    y,
                    r,
                    g,
                    b,
                    a,
                    gray_value
                ])

    print(
        "Pixel CSV saved:",
        csv_path
    )


# ============================================================
# RENDER + PIXEL EXTRACTION
# ============================================================

def render_and_extract(
    filename
):

    filepath = os.path.join(
        OUTPUT_FOLDER,
        filename
    )

    scene = bpy.context.scene

    scene.render.filepath = filepath

    print(
        "\nRendering:",
        filename
    )

    bpy.ops.render.render(
        write_still=True
    )

    # --------------------------------------------------------
    # Get rendered image from Blender
    # --------------------------------------------------------

    image = bpy.data.images.get(
        "Render Result"
    )

    if image is None:

        raise RuntimeError(
            "Render Result not found."
        )

    # --------------------------------------------------------
    # Extract pixel values
    # --------------------------------------------------------

    save_pixel_values(
        filename,
        image
    )

    print(
        "Finished:",
        filename
    )


# ============================================================
# GRAYSCALE DATASET
# ============================================================

print("\n========================================")
print("GENERATING GRAYSCALE DATASET")
print("========================================")

set_grayscale()


for pose_name, rotation_z in poses.items():

    # --------------------------------------------------------
    # Reset animal location
    # --------------------------------------------------------

    nilgai.location = (
        original_location.copy()
    )

    # --------------------------------------------------------
    # Reset animal rotation
    # --------------------------------------------------------

    nilgai.rotation_euler = (
        original_rotation.copy()
    )

    # --------------------------------------------------------
    # Change ONLY animal orientation
    # --------------------------------------------------------

    nilgai.rotation_euler.z = (
        original_rotation.z
        + rotation_z
    )

    # --------------------------------------------------------
    # Render
    # --------------------------------------------------------

    render_and_extract(
        "grayscale_"
        + pose_name
        + ".png"
    )


# ============================================================
# IRONBOW DATASET
# ============================================================

print("\n========================================")
print("GENERATING IRONBOW DATASET")
print("========================================")

set_ironbow()


for pose_name, rotation_z in poses.items():

    # --------------------------------------------------------
    # Reset animal location
    # --------------------------------------------------------

    nilgai.location = (
        original_location.copy()
    )

    # --------------------------------------------------------
    # Reset animal rotation
    # --------------------------------------------------------

    nilgai.rotation_euler = (
        original_rotation.copy()
    )

    # --------------------------------------------------------
    # Change ONLY animal orientation
    # --------------------------------------------------------

    nilgai.rotation_euler.z = (
        original_rotation.z
        + rotation_z
    )

    # --------------------------------------------------------
    # Render
    # --------------------------------------------------------

    render_and_extract(
        "ironbow_"
        + pose_name
        + ".png"
    )


# ============================================================
# RESTORE ORIGINAL MODEL
# ============================================================

nilgai.location = (
    original_location
)

nilgai.rotation_euler = (
    original_rotation
)

nilgai.scale = (
    original_scale
)


# ============================================================
# FINISHED
# ============================================================

print("\n========================================")
print("PIXEL DATASET GENERATION COMPLETE")
print("========================================")

print(
    "Output folder:"
)

print(
    OUTPUT_FOLDER
)

print("\nGenerated:")

for pose_name in poses.keys():

    print(
        "grayscale_"
        + pose_name
        + ".png"
    )

    print(
        "grayscale_"
        + pose_name
        + "_pixels.csv"
    )


for pose_name in poses.keys():

    print(
        "ironbow_"
        + pose_name
        + ".png"
    )

    print(
        "ironbow_"
        + pose_name
        + "_pixels.csv"
    )

print("========================================")

## yolo ds 100 img save 0-50

In [ ]:
import bpy
import os
import math
import colorsys
import random

# ============================================================
# FOLDER STRUCTURE (OUTPUT: yolo_ds)
# ============================================================
BASE_DIR = os.path.join(bpy.path.abspath("//"), "yolo_ds22")
DIR_WHITE_HOT = os.path.join(BASE_DIR, "white_hot")
DIR_IRONBOW_RGB = os.path.join(BASE_DIR, "ironbow_rgb")
DIR_IRONBOW_HSV = os.path.join(BASE_DIR, "ironbow_hsv")
DIR_LABELS = os.path.join(BASE_DIR, "labels")

for folder in [BASE_DIR, DIR_WHITE_HOT, DIR_IRONBOW_RGB, DIR_IRONBOW_HSV, DIR_LABELS]:
    os.makedirs(folder, exist_ok=True)

bpy.context.scene.view_settings.view_transform = 'Standard'

# ============================================================
# FIND OBJECTS
# ============================================================
soil = bpy.data.objects.get("SOIL_GROUND")
nilgai = bpy.data.objects.get("mesh_0")

grass_objects = []
for obj in bpy.context.scene.objects:
    if obj.type == 'MESH' and obj.name.startswith("Object_2"):
        grass_objects.append(obj)

if soil is None or nilgai is None:
    raise RuntimeError("Run 01_GUI_Main_Scene.py first to setup objects.")

# Force standard XYZ rotation mode so script rotations apply correctly
nilgai.rotation_mode = 'XYZ'

# ============================================================
# EXACT DOCUMENT LOOKUP TABLE VALUES
# ============================================================
VAL_WHITE_HOT = {
    "mammal": (0.576, 0.576, 0.576, 1.0),
    "grass":  (0.502, 0.502, 0.502, 1.0),
    "soil":   (0.502, 0.502, 0.502, 1.0),
    "sky":    (0.000, 0.000, 0.000, 1.0)
}

VAL_IRONBOW_RGB = {
    "mammal": (0.945, 0.400, 0.012, 1.0),
    "grass":  (0.906, 0.278, 0.082, 1.0),
    "soil":   (0.906, 0.278, 0.082, 1.0),
    "sky":    (0.000, 0.000, 0.039, 1.0)
}

VAL_MASK = {
    "mammal": (1.0, 1.0, 1.0, 1.0),
    "bg":     (0.0, 0.0, 0.0, 1.0) 
}

# ============================================================
# MATERIAL CREATOR
# ============================================================
def create_flat_material(name, color):
    mat = bpy.data.materials.get(name)
    if mat is None:
        mat = bpy.data.materials.new(name)
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    nodes.clear()
    
    node_emission = nodes.new(type='ShaderNodeEmission')
    node_emission.inputs['Color'].default_value = color
    node_emission.inputs['Strength'].default_value = 1.0
    
    node_output = nodes.new(type='ShaderNodeOutputMaterial')
    links.new(node_emission.outputs['Emission'], node_output.inputs['Surface'])
    return mat

def assign_material(obj, material):
    if obj is None or material is None: return
    if len(obj.data.materials) == 0:
        obj.data.materials.append(material)
    else:
        obj.data.materials[0] = material

def set_scene_materials(dataset_type):
    if dataset_type == "MASK":
        mat_animal = create_flat_material("MASK_ANIMAL", VAL_MASK["mammal"])
        mat_bg = create_flat_material("MASK_BG", VAL_MASK["bg"])
        sky_color = VAL_MASK["bg"]
        
        assign_material(nilgai, mat_animal)
        assign_material(soil, mat_bg)
        for obj in grass_objects:
            assign_material(obj, mat_bg)

    elif dataset_type == "WHITE_HOT":
        mat_animal = create_flat_material("WH_ANIMAL", VAL_WHITE_HOT["mammal"])
        mat_grass = create_flat_material("WH_GRASS", VAL_WHITE_HOT["grass"])
        mat_soil = create_flat_material("WH_SOIL", VAL_WHITE_HOT["soil"])
        sky_color = VAL_WHITE_HOT["sky"]
        
        assign_material(nilgai, mat_animal)
        assign_material(soil, mat_soil)
        for obj in grass_objects:
            assign_material(obj, mat_grass)

    elif dataset_type == "IRONBOW":
        mat_animal = create_flat_material("IB_ANIMAL", VAL_IRONBOW_RGB["mammal"])
        mat_grass = create_flat_material("IB_GRASS", VAL_IRONBOW_RGB["grass"])
        mat_soil = create_flat_material("IB_SOIL", VAL_IRONBOW_RGB["soil"])
        sky_color = VAL_IRONBOW_RGB["sky"]
        
        assign_material(nilgai, mat_animal)
        assign_material(soil, mat_soil)
        for obj in grass_objects:
            assign_material(obj, mat_grass)

    world = bpy.context.scene.world
    if world and world.use_nodes:
        bg_node = world.node_tree.nodes.get('Background')
        if bg_node:
            bg_node.inputs['Color'].default_value = sky_color

# ============================================================
# ACCURATE 2D SILHOUETTE BBOX EXTRACTION
# ============================================================
def extract_mask_bbox(pixels, w, h):
    min_x, max_x = w, -1
    min_y, max_y = h, -1
    found = False

    for i in range(0, len(pixels), 4):
        if pixels[i] > 0.5:
            found = True
            pixel_index = i // 4
            x = pixel_index % w
            y = pixel_index // w
            
            if x < min_x: min_x = x
            if x > max_x: max_x = x
            if y < min_y: min_y = y
            if y > max_y: max_y = y

    if not found:
        return None

    yolo_w = (max_x - min_x) / w
    yolo_h = (max_y - min_y) / h
    yolo_x = (min_x + max_x) / 2.0 / w
    yolo_y = (h - (min_y + max_y) / 2.0) / h

    return (yolo_x, yolo_y, yolo_w, yolo_h)

# ============================================================
# RENDER LOGIC
# ============================================================
original_location = nilgai.location.copy()
original_rotation = nilgai.rotation_euler.copy()

def render_frame(pose_name, offset_x, offset_y, rot_z):
    scene = bpy.context.scene
    
    nilgai.location.x = original_location.x + offset_x
    nilgai.location.y = original_location.y + offset_y
    # Directly assign the exact random rotation to the Z axis (yaw)
    nilgai.rotation_euler[2] = rot_z 
    
    # Force Blender to update the 3D geometry before masking
    bpy.context.view_layer.update()
    
    # --------------------------------------------------------
    # 1. RENDER SILHOUETTE MASK & EXTRACT BBOX
    # --------------------------------------------------------
    set_scene_materials("MASK")
    mask_filepath = os.path.join(BASE_DIR, "temp_mask.png")
    scene.render.filepath = mask_filepath
    bpy.ops.render.render(write_still=True)
    
    mask_img = bpy.data.images.load(mask_filepath)
    w, h = mask_img.size[0], mask_img.size[1]
    mask_pixels = list(mask_img.pixels)
    
    bbox = extract_mask_bbox(mask_pixels, w, h)
    
    bpy.data.images.remove(mask_img)
    if os.path.exists(mask_filepath):
        os.remove(mask_filepath)
        
    if bbox:
        with open(os.path.join(DIR_LABELS, f"{pose_name}.txt"), "w") as f:
            f.write(f"0 {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
    else:
        print(f"[{pose_name}] Nilgai is 100% hidden. No bounding box generated.")

    # --------------------------------------------------------
    # 2. RENDER WHITE HOT
    # --------------------------------------------------------
    set_scene_materials("WHITE_HOT")
    wh_filename = f"white_hot_{pose_name}.png"
    scene.render.filepath = os.path.join(DIR_WHITE_HOT, wh_filename)
    bpy.ops.render.render(write_still=True)
    
    # --------------------------------------------------------
    # 3. RENDER IRONBOW RGB & SAVE HSV IMAGE
    # --------------------------------------------------------
    set_scene_materials("IRONBOW")
    ib_filename = f"ironbow_rgb_{pose_name}.png"
    ib_filepath = os.path.join(DIR_IRONBOW_RGB, ib_filename)
    scene.render.filepath = ib_filepath
    bpy.ops.render.render(write_still=True)
    
    ib_img = bpy.data.images.load(ib_filepath)
    ib_pixels = list(ib_img.pixels)
    hsv_image_pixels = []
    
    for i in range(0, len(ib_pixels), 4):
        r, g, b = ib_pixels[i], ib_pixels[i+1], ib_pixels[i+2]
        h_fl, s_fl, v_fl = colorsys.rgb_to_hsv(r, g, b)
        hsv_image_pixels.extend([h_fl, s_fl, v_fl, 1.0])

    hsv_img_name = f"ironbow_hsv_{pose_name}"
    hsv_img = bpy.data.images.get(hsv_img_name)
    if hsv_img:
        bpy.data.images.remove(hsv_img)
        
    hsv_img = bpy.data.images.new(hsv_img_name, width=w, height=h)
    hsv_img.pixels = hsv_image_pixels
    hsv_img.filepath_raw = os.path.join(DIR_IRONBOW_HSV, f"ironbow_hsv_{pose_name}.png")
    hsv_img.file_format = 'PNG'
    hsv_img.save()
    
    bpy.data.images.remove(ib_img)
    bpy.data.images.remove(hsv_img)

    print(f"Generated frame: {pose_name}")

# ============================================================
# RUN 100 RANDOMIZED FRAMES
# ============================================================
print("\n========================================")
print("STARTING 100-FRAME MASKED DATASET GENERATION")
print("========================================")

TOTAL_FRAMES = 100

for frame in range(TOTAL_FRAMES):
    pose_name = f"frame_{frame:04d}"
    
    rand_x = random.uniform(-1.5, 1.5)
    rand_y = random.uniform(-1.5, 1.5)
    
    # Generate a random angle between 0 and 360 degrees (in radians)
    rand_rot_z = random.uniform(0, math.pi * 2)
    
    render_frame(pose_name, rand_x, rand_y, rand_rot_z)

# Restore Original Position and Rotation
nilgai.location = original_location
nilgai.rotation_euler = original_rotation

print("\n========================================")
print("SUCCESS: 100 FRAMES SAVED TO yolo_ds")
print("========================================")